# Chapter 40 — Evaluating AI Systems Without a Right Answer

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch40/_lib.py`.

In [2]:
import numpy as np, warnings; warnings.filterwarnings("ignore")

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# When there is no single correct answer, human judgment becomes the
# reference, and human judgment has its own reliability to measure.
# Simulate raters against a KNOWN true quality, so agreement can be
# checked against ground truth rather than only against each other.
r = np.random.default_rng(40)
n_items = 300
true_quality = r.uniform(1, 5, n_items)          # a hidden "true" score, 1-5

def simulate_rater(true_scores, noise_sd, seed):
    rr = np.random.default_rng(seed)
    noisy = true_scores + rr.normal(0, noise_sd, len(true_scores))
    return np.clip(np.round(noisy), 1, 5).astype(int)

def cohen_kappa(a, b, n_categories=5):
    po = np.mean(a == b)
    pe = sum(np.mean(a == k) * np.mean(b == k) for k in range(1,
             n_categories + 1))
    return (po - pe) / (1 - pe)

print(f"{'rater noise (sd)':>18}{'raw agreement':>16}{'cohen kappa':>14}")
for noise in (0.2, 0.6, 1.0, 1.5, 2.5):
    agrees, kappas = [], []
    for pair_seed in range(20):     # average over 20 independent rater pairs
        rater_a = simulate_rater(true_quality, noise, seed=1000 + pair_seed)
        rater_b = simulate_rater(true_quality, noise, seed=2000 + pair_seed)
        agrees.append(np.mean(rater_a == rater_b))
        kappas.append(cohen_kappa(rater_a, rater_b))
    print(f"{noise:>18.1f}{np.mean(agrees):>16.4f}{np.mean(kappas):>14.4f}")

  rater noise (sd)   raw agreement   cohen kappa
               0.2          0.7720        0.7082
               0.6          0.4540        0.3069
               1.0          0.3367        0.1678
               1.5          0.2837        0.1025
               2.5          0.2743        0.0575


### Block 2  (`c2.py`)

In [4]:
# A rubric splits one holistic judgment into several specific criteria,
# then averages them. If each criterion's noise is at least partly
# independent of the others, averaging reduces the total noise, the
# same averaging argument Chapter 20 made about ensembles.
def simulate_holistic(true_scores, noise_sd, seed):
    rr = np.random.default_rng(seed)
    return true_scores + rr.normal(0, noise_sd, len(true_scores))

def simulate_rubric(true_scores, noise_sd, n_criteria, seed):
    rr = np.random.default_rng(seed)
    criteria_scores = np.stack([
        true_scores + rr.normal(0, noise_sd, len(true_scores))
        for _ in range(n_criteria)
    ])
    return criteria_scores.mean(axis=0)        # the rubric score: an average

print(f"{'method':<28}{'error vs true quality (RMSE)':>30}")
holistic = simulate_holistic(true_quality, noise_sd=1.0, seed=40)
rmse_holistic = np.sqrt(np.mean((holistic - true_quality) ** 2))
print(f"{'single holistic rating':<28}{rmse_holistic:>30.4f}")

for n_criteria in (2, 3, 5, 10):
    rubric = simulate_rubric(true_quality, noise_sd=1.0,
                             n_criteria=n_criteria, seed=40)
    rmse_rubric = np.sqrt(np.mean((rubric - true_quality) ** 2))
    print(f"{'rubric, ' + str(n_criteria) + ' criteria':<28}"
          f"{rmse_rubric:>30.4f}")

print(f"\neach criterion is exactly as noisy as the holistic rating alone;")
print(f"only averaging several of them, not any single criterion, reduces")
print(f"the error.")

method                        error vs true quality (RMSE)
single holistic rating                              1.0620
rubric, 2 criteria                                  0.7638
rubric, 3 criteria                                  0.5898
rubric, 5 criteria                                  0.4420
rubric, 10 criteria                                 0.3014

each criterion is exactly as noisy as the holistic rating alone;
only averaging several of them, not any single criterion, reduces
the error.


### Block 3  (`c3.py`)

In [5]:
# A toy judge, built to be simple enough to inspect completely: it
# scores a response by a mix of length and vocabulary overlap with the
# question, then picks a winner between two candidates. This is not a
# language model. It is simple enough that its own biases can be shown
# in full, which is the point: the same class of bias is documented in
# real LLM judges by the papers cited at the end of this chapter, and
# a fully transparent toy version makes the mechanism visible rather
# than asserted.
def toy_judge_score(response, question_words):
    words = response.lower().split()
    overlap = len(set(words) & question_words)
    length_bonus = len(words) * 0.15    # the deliberate flaw: rewards length
    return overlap + length_bonus

def toy_judge_compare(resp_a, resp_b, question):
    qwords = set(question.lower().split())
    score_a = toy_judge_score(resp_a, qwords)
    score_b = toy_judge_score(resp_b, qwords)
    return "A" if score_a > score_b else "B"

question = "what causes the seasons to change on Earth"
short_correct = ("Earth's tilted axis changes how directly sunlight hits "
                 "each hemisphere through the year.")
long_padded = ("Well, that's a great question, and there are actually many "
               "interesting factors to " +
              "consider here, but if we think about it carefully and look "
              "at the science, the main " +
              "thing going on is really about how Earth's axis is tilted "
              "relative to its orbit.")

winner_1 = toy_judge_compare(short_correct, long_padded, question)
# same pair, order swapped
winner_2 = toy_judge_compare(long_padded, short_correct, question)

print(f"short, correct answer:  {len(short_correct.split())} words")
print(f"long, padded answer:    {len(long_padded.split())} words, same core "
      f"claim, wrapped in filler")
print(f"\npresented as (A=short, B=long): judge prefers {winner_1}")
print(f"presented as (A=long, B=short): judge prefers {winner_2}")

short, correct answer:  13 words
long, padded answer:    44 words, same core claim, wrapped in filler

presented as (A=short, B=long): judge prefers B
presented as (A=long, B=short): judge prefers A


### Block 4  (`c4.py`)

In [6]:
# Position bias is a different failure from length bias: even holding
# content fixed, a judge can favour whichever answer sits in a
# particular slot. This toy judge scores both responses identically on
# content and adds a small, fixed preference for whichever is shown
# first, exactly the kind of subtle artifact documented in real judge
# models trained on data where the first option happened to be
# correct slightly more often.
def biased_judge_compare(resp_a, resp_b, question, position_bonus=0.4):
    qwords = set(question.lower().split())
    # "shown first" bonus
    score_a = toy_judge_score(resp_a, qwords) + position_bonus
    score_b = toy_judge_score(resp_b, qwords)
    return "A" if score_a > score_b else "B"

resp_1 = ("The seasonal cycle is driven by the tilt of Earth's "
          "rotational axis.")
resp_2 = ("Earth's axial tilt changes the angle of incoming sunlight "
          "across the year.")

print("two responses of near-identical length and content:")
print(f"  resp_1: {len(resp_1.split())} words")
print(f"  resp_2: {len(resp_2.split())} words")

win_a_first = biased_judge_compare(resp_1, resp_2, question)
win_b_first = biased_judge_compare(resp_2, resp_1, question)
print(f"\nshown as (A=resp_1, B=resp_2): judge picks {win_a_first}")
print(f"shown as (A=resp_2, B=resp_1): judge picks {win_b_first}")
print(f"\nthe judge picked whichever response was shown in "
      f"slot A both times,")
print(f"despite the two responses being interchangeable in "
      f"content and length.")

# quantify how often this happens across many near-tied response pairs
r_bias = np.random.default_rng(40)
flips = 0
for _ in range(200):
    len_a = r_bias.integers(10, 15)
    len_b = r_bias.integers(10, 15)
    fake_a = " ".join(["word"] * len_a)
    fake_b = " ".join(["word"] * len_b)
    w1 = biased_judge_compare(fake_a, fake_b, question)
    w2 = biased_judge_compare(fake_b, fake_a, question)
    # a consistent judge would pick the SAME underlying response both times;
    # this judge picks "A" both times, i.e. whichever is shown first
    if w1 == "A" and w2 == "A":
        flips += 1
print(f"\nacross 200 near-tied pairs, the judge preferred slot A regardless")
print(f"of content in {flips} of 200 cases ({100*flips/200:.0f}%).")

two responses of near-identical length and content:
  resp_1: 12 words
  resp_2: 12 words

shown as (A=resp_1, B=resp_2): judge picks A
shown as (A=resp_2, B=resp_1): judge picks A

the judge picked whichever response was shown in slot A both times,
despite the two responses being interchangeable in content and length.

across 200 near-tied pairs, the judge preferred slot A regardless
of content in 157 of 200 cases (78%).


### Block 5  (`c5.py`)

In [7]:
# The standard mitigation for position bias costs exactly double the
# judging calls: present the same pair in both orders, and treat a
# disagreement between the two runs as "no clear winner" rather than
# trusting whichever order happened to be used.
def debiased_compare(resp_a, resp_b, question):
    verdict_1 = biased_judge_compare(resp_a, resp_b, question)
    verdict_2 = biased_judge_compare(resp_b, resp_a, question)
    winner_1 = resp_a if verdict_1 == "A" else resp_b
    winner_2 = resp_b if verdict_2 == "A" else resp_a
    if winner_1 == winner_2:
        return winner_1, True            # both orders agree: a real signal
    return None, False                   # disagreement: flag it, don't guess

resolved, flagged = 0, 0
for _ in range(200):
    len_a = r_bias.integers(10, 15)
    len_b = r_bias.integers(10, 15)
    fake_a = " ".join(["word"] * len_a)
    fake_b = " ".join(["word"] * len_b)
    winner, agreed = debiased_compare(fake_a, fake_b, question)
    if agreed:
        resolved += 1
    else:
        flagged += 1

print(f"of another 200 near-tied pairs, checking both orders:")
print(f"  agreed on a genuine winner: {resolved}")
print(f"  flagged as no clear winner (orders disagreed): {flagged}")
print(f"\nthe single-order judge from Step 4 confidently declared a winner")
print(f"in all 200 cases, 157 of them driven purely by position. Checking")
print(f"both orders converts most of that false confidence into an honest")
print(f"'no clear winner,' at twice the cost in judging calls.")

of another 200 near-tied pairs, checking both orders:
  agreed on a genuine winner: 99
  flagged as no clear winner (orders disagreed): 101

the single-order judge from Step 4 confidently declared a winner
in all 200 cases, 157 of them driven purely by position. Checking
both orders converts most of that false confidence into an honest
'no clear winner,' at twice the cost in judging calls.
